### New algorithm for enumerating every $\Z_2^n$-colorable PL~spheres of Picard number $5$

In [1]:
using Oscar
using ProgressBars
using SparseArrays
using Serialization
using Base.Threads

println("Running on $(Threads.nthreads()) threads")



Running on 10 threads


In [2]:
function boundary_incidence_facets_to_ridges(facets::Vector{UInt16})
    # d = popcount(facets[1]) - 1  # facet dimension
    # @assert all(popcount(f) == d+1 for f in facets) "Need a pure complex (all facets same size)."

    # collect ridges (each facet contributes its (d-1)-subfaces by deleting one vertex)
    ridge_dict = Dict{UInt16, Int}()  # ridge -> row index
    ridges = Vector{UInt16}()
    for f in facets
        for i=0:((8*sizeof(f))-1)
            if (f>>i)&1==1
                r = f ⊻ (UInt16(1)<<i)
                if !haskey(ridge_dict, r)
                    push!(ridges, r)
                    ridge_dict[r] = length(ridges)
                end
            end
        end
    end

    m = length(ridges)
    n = length(facets)

    # build sparse 0/1 matrix (m x n): ridges x facets
    I = Int[]   # row indices
    J = Int[]   # col indices
    V = Int8[]  # values (all ones)

    for (j, f) in pairs(facets)
        for i=0:(8*sizeof(f)-1)
            if (f>>i)&1==1
                r = f ⊻ (UInt16(1)<<i)
                i = ridge_dict[r]
                push!(I, i)
                push!(J, j)
                push!(V, true)
            end
        end
    end

    A = sparse(I, J, V, m, n)  # SparseMatrixCSC{Bool} 

    return ridges, A
end



boundary_incidence_facets_to_ridges (generic function with 1 method)

In [3]:
function kernel_basis_mod2_sparse(A)
    m, n = size(A)

    # Store rows as sparse BitVectors (using sets of column indices)
    rows = [Set{Int}() for _ in 1:m]
    
    # Build rows from A mod 2
    @inbounds for j in 1:n
        for p in nzrange(A, j)
            i = rowvals(A)[p]
            if isodd(A.nzval[p])
                if j in rows[i]
                    delete!(rows[i], j)  # XOR: 1⊕1=0
                else
                    push!(rows[i], j)     # XOR: 0⊕1=1
                end
            end
        end
    end

    # RREF over GF(2), recording pivot columns and pivot rows
    pivcol = Int[]
    pivrow = Int[]
    row = 1
    
    @inbounds for col in 1:n
        row > m && break

        # Find pivot in this column at/under current row
        piv = 0
        for r in row:m
            if col in rows[r]
                piv = r
                break
            end
        end
        piv == 0 && continue

        # Swap rows
        if piv != row
            rows[row], rows[piv] = rows[piv], rows[row]
        end

        push!(pivcol, col)
        push!(pivrow, row)

        # Eliminate this column in all other rows (RREF)
        pivot_row = rows[row]
        for r in 1:m
            if r != row && col in rows[r]
                # XOR rows[r] with pivot_row
                symdiff!(rows[r], pivot_row)
            end
        end

        row += 1
    end

    # Free columns
    pivot_set = Set(pivcol)
    free_cols = [j for j in 1:n if !(j in pivot_set)]

    basis = BitVector[]
    isempty(free_cols) && return basis

    # Build one kernel vector per free column
    @inbounds for f in free_cols
        x_set = Set{Int}([f])  # Sparse representation of x

        # Back-substitute using RREF rows
        for t in length(pivcol):-1:1
            c = pivcol[t]
            r = pivrow[t]
            
            # Compute parity: count elements in rows[r] ∩ x_set, excluding c
            row_r = rows[r]
            parity = false
            for j in row_r
                if j != c && j in x_set
                    parity = !parity
                end
            end
            
            # Set x[c] based on parity
            if parity
                push!(x_set, c)
            else
                delete!(x_set, c)
            end
        end

        # Convert sparse set to BitVector
        x = falses(n)
        for j in x_set
            x[j] = true
        end
        push!(basis, x)
    end

    return basis
end

kernel_basis_mod2_sparse (generic function with 1 method)

In [4]:
using SparseArrays

"""
    kernel_elements_with_Ax_in_0_or_2_noprune(A, basis) -> Vector{BitVector}

Full enumeration WITHOUT pruning.

- A is strictly 0/1 and used over ℤ for Ax.
- basis is a GF(2)-basis of ker(A mod 2), given as Vector{BitVector}.
- Enumerates all nonzero x in the GF(2)-span of `basis` and returns those with Ax ∈ {0,2}^m.
- Complexity: Θ(2^k * (cost to update/check)), where k = length(basis).
"""
function kernel_elements_with_Ax_in_0_or_2(
    A::SparseMatrixCSC{<:Integer},
    basis::Vector{BitVector},
)
    m, n = size(A)
    k = length(basis)
    # println("Dimension of the kernel: ",k)
    @assert all(length(b) == n for b in basis) "Basis vectors must have length = #cols(A)."

    # column -> incident rows (A is 0/1)
    rv = rowvals(A)
    colrows = Vector{Vector{Int}}(undef, n)
    for j in 1:n
        colrows[j] = [rv[p] for p in nzrange(A, j)]
    end

    # supports of basis vectors (columns they toggle)
    supp = Vector{Vector{Int}}(undef, k)
    for t in 1:k
        supp[t] = findall(basis[t])
    end

    x = falses(n)
    counts = zeros(Int16, m)  # Ax over ℤ
    sols = BitVector[]

    function toggle_basis!(t::Int)
        @inbounds for j in supp[t]
            turning_on = !x[j]
            x[j] = turning_on
            δ = turning_on ? Int16(1) : Int16(-1)
            for i in colrows[j]
                counts[i] += δ
            end
        end
    end

    @inline function feasible_final()::Bool
        @inbounds for i in 1:m
            c = counts[i]
            if !(c == 0 || c == 2)
                return false
            end
        end
        return true
    end

    function dfs(t::Int, any_one::Bool)
        if t > k
            if any_one && feasible_final()
                push!(sols, copy(x))
            end
            return
        end

        # branch 0: do not include basis[t]
        dfs(t + 1, any_one)

        # branch 1: include basis[t]
        toggle_basis!(t)
        dfs(t + 1, true)
        toggle_basis!(t)  # toggle back (self-inverse over GF(2))
    end

    dfs(1, false)  # excludes zero vector by any_one=false
    return sols
end


kernel_elements_with_Ax_in_0_or_2

In [12]:
# If homology is UNREDUCED in your setup (most common):
function is_homology_sphere(K::Oscar.SimplicialComplex)
    b = betti_numbers(K)
    d = dim(K)
    for i in 0:d
        if i == 0 || i == d
            b[i+1] == 1 || return false
        else
            b[i+1] == 0 || return false
        end
    end
    return true
end


is_homology_sphere (generic function with 1 method)

In [ ]:
using BenchmarkTools


function kernel_basis_echelon_prioritize(B, S)
    """
    Convert kernel basis B to echelon form over GF(2), prioritizing columns in S.
    Returns (B_ech, pivots) where:
    - B_ech is the echelon basis
    - pivots[i] is the pivot position (leading 1) of B_ech[i]
    - Columns in S are processed first to maximize forced coefficients
    """
    isempty(B) && return (BitVector[], Int[])
    
    n = length(B[1])
    k = length(B)
    
    # Copy basis vectors
    B_ech = [copy(b) for b in B]
    pivots = Int[]
    
    # Build column order: S columns first, then others
    cols_in_S = [j for j in 1:n if S[j]]
    cols_not_in_S = [j for j in 1:n if !S[j]]
    col_order = vcat(cols_in_S, cols_not_in_S)
    
    current_row = 1
    for col in col_order
        current_row > k && break
        
        # Find a vector with 1 at position col (at or below current_row)
        piv = 0
        for r in current_row:k
            if B_ech[r][col]
                piv = r
                break
            end
        end
        
        piv == 0 && continue  # No pivot in this column
        
        # Swap to current position
        if piv != current_row
            B_ech[current_row], B_ech[piv] = B_ech[piv], B_ech[current_row]
        end
        
        push!(pivots, col)
        
        # Eliminate this column in all other rows
        for r in 1:k
            if r != current_row && B_ech[r][col]
                B_ech[r] .⊻= B_ech[current_row]
            end
        end
        
        current_row += 1
    end
    
    return (B_ech, pivots)
end


function enumerate_kernel_with_constraints(A, B, S)
    m, n = size(A)
    results = BitVector[]
    
    if isempty(B)
        x = falses(n)
        if all(.!S) && check_Ax_is_02(A, x)
            push!(results, x)
        end
        return results
    end
    
    # Convert basis to echelon form, prioritizing S columns
    B_ech, pivots = kernel_basis_echelon_prioritize(B, S)
    k = length(B_ech)
    
    # Determine forced and free coefficients
    forced_lambda = falses(k)
    free_indices = Int[]
    
    for i in 1:k
        piv = pivots[i]
        if S[piv]
            forced_lambda[i] = true
        else
            push!(free_indices, i)
        end
    end
    
    # Compute base vector from forced coefficients
    x_forced = falses(n)
    for i in 1:k
        if forced_lambda[i]
            x_forced .⊻= B_ech[i]
        end
    end
    
    # Early exit if x_forced doesn't satisfy S
    for j in 1:n
        if S[j] && !x_forced[j]
            return results
        end
    end
    
    num_free = length(free_indices)
    
    # Early exit if no free variables
    if num_free == 0
        if check_Ax_is_02(A, x_forced)
            push!(results, copy(x_forced))
        end
        return results
    end
    
    # Pre-allocate with size hint
    sizehint!(results, min(2^num_free, 1000))
    
    # Gray code enumeration
    x = copy(x_forced)
    
    # First iteration (all free vars = 0)
    if check_Ax_is_02_parallel(A, x)
        push!(results, copy(x))
    end
    
    # Enumerate using Gray code
    for i in 1:(2^num_free - 1)
        gray = i ⊻ (i >> 1)
        gray_prev = (i - 1) ⊻ ((i - 1) >> 1)
        changed_bit = trailing_zeros(gray ⊻ gray_prev) + 1
        
        idx = free_indices[changed_bit]
        x .⊻= B_ech[idx]
        
        if check_Ax_is_02_parallel(A, x)
            push!(results, copy(x))
        end
    end
    
    return results
end

function check_Ax_is_02(A, x)
    m, n = size(A)
    result = zeros(Int, m)
    
    for j in 1:n
        if x[j]
            for p in nzrange(A, j)
                i = rowvals(A)[p]
                result[i] += A.nzval[p]
                
                if result[i] > 2
                    return false
                end
            end
        end
    end
    
    return true

end



function check_Ax_is_02_parallel(A, x)
    m, n = size(A)
    nthreads = Threads.nthreads()
    
    # Each thread gets its own result vector
    thread_results = [zeros(Int, m) for _ in 1:nthreads]
    thread_failed = falses(nthreads)
    
    @threads for j in 1:n
        tid = Threads.threadid()
        
        # Check if any thread has failed
        any(thread_failed) && continue
        
        if x[j]
            for p in nzrange(A, j)
                i = rowvals(A)[p]
                thread_results[tid][i] += A.nzval[p]
                
                if thread_results[tid][i] > 2
                    thread_failed[tid] = true
                    break
                end
            end
        end
    end
    
    # Early exit if any thread failed
    any(thread_failed) && return false
    
    # Combine results and check
    result = zeros(Int, m)
    for t in 1:nthreads
        result .+= thread_results[t]
        # Check during combining
        for i in 1:m
            if result[i] > 2
                return false
            end
        end
    end
    
    return true
end

check_Ax_is_02_parallel (generic function with 1 method)

In [ ]:
using Profile
using BenchmarkTools
using ProgressMeter

global mat_DB_bin = open("rank_4_simple_bin_mat_DB_bin.jls", "r") do io
    deserialize(io)
end

global iso_DB = open("rank_4_iso_DB_m_7-15.jls", "r") do io
    deserialize(io)
end

function subset_bitvector(superset::Vector{UInt16}, subset::Vector{UInt16})
    n = length(superset)
    result = falses(n)
    
    j = 1  # index dans subset
    for i in 1:n
        if j <= length(subset) && superset[i] == subset[j]
            result[i] = true
            j += 1
        end
    end
    
    return result
end




function build_finalDB_single_v!(pseudo_manifolds_DB::Dict{Int,Vector{Set{BitVector}}},mat_DB::Dict{Int,Vector{Vector{UInt16}}},iso_DB::Dict{Int,Dict{Int,Tuple{Int,Int,Any}}},mmax)
    mmin = minimum(collect(keys(mat_DB)))
    for m=mmin:mmax
        println(m)
        pseudo_manifolds_DB[m] = Vector{Set{BitVector}}()
        for (l,bases) in enumerate(mat_DB[m])
            # display(bases)
            V = reduce(|,bases)
            compl_bases = [base⊻V for base in bases] # we need to complement to get the correct boundary matrix
            ridges, A = boundary_incidence_facets_to_ridges(compl_bases)  
            basis = kernel_basis_mod2_sparse(A)
            push!(pseudo_manifolds_DB[m], Set{BitVector}())
            if m==mmin
                mandatory_facets_bit=falses(length(bases))
                all_solutions_bit = kernel_elements_with_Ax_in_0_or_2(A,basis)
                for K_bit in all_solutions_bit
                    K = simplicial_complex(collect([i for i=1:16 if facet>>(i-1)&1==1] for facet in compl_bases[findall(K_bit)]))
                    if is_sphere(K)
                        union!(pseudo_manifolds_DB[m][l],Set([K_bit]))
                    end
                end
            else
                index_contraction, v_contract, perm = iso_DB[m][l]
                @showprogress for L in pseudo_manifolds_DB[m-1][index_contraction]
                    mandatory_facets = [reduce(|,[UInt16(1)<<(perm[i]-1) for i=1:(sizeof(facet_L)) if (facet_L>>(i-1)&1)==1],init=UInt16(0))⊻(UInt16(1)<<(v_contract-1)) for facet_L in mat_DB[m-1][index_contraction][findall(L)]]
                    # print(mandatory_facets)
                    mandatory_facets_bit = subset_bitvector(bases, mandatory_facets)
                    t1 = time()
                    all_solutions_bit = enumerate_kernel_with_constraints(A,basis,mandatory_facets_bit)
                    if (time() - t1)> 1
                        println("Enum: ", time() - t1, " seconds")
                    end

                    # for K_bit in all_solutions_bit
                    #     K = simplicial_complex(collect(compl_bases[findall(K_bit)]))
                    #     if is_homology_sphere(K)
                    #         test = true
                    #         for v in vertex_list_of_bases([collect(facet) for facet in facets(K)])
                    #             if !is_homology_sphere(link_subcomplex(K,[v]))
                    #                 test=false
                    #                 break
                    #             end
                    #         end
                    #         if test
                    #             push!(pseudo_manifolds_DB[m][l],K_bit)
                    #         end
                    #     end
                    # end
                    union!(pseudo_manifolds_DB[m][l],all_solutions_bit)
                end
            end
        end
    end
end


global pseudo_manifolds_DB = Dict{Int,Vector{Set{BitVector}}}()

build_finalDB_single_v!(pseudo_manifolds_DB,mat_DB_bin,iso_DB,15)

# using ProfileView
# ProfileView.view()




6
7


Progress: 100%|█████████████████████████████████████████| Time: 0:00:00


8


Progress: 100%|█████████████████████████████████████████| Time: 0:00:00
Progress:  16%|██████▌                                  |  ETA: 0:02:41